In [19]:
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

In [5]:
PROJECT_ROOT = Path.cwd()

CSV_PATH = PROJECT_ROOT / "data" / "raw" / "Resume.csv"
PDF_DIR = PROJECT_ROOT  / "data" / "raw" / "pdfs"

In [6]:
print(Path.cwd())

c:\Users\swind\OneDrive\Desktop\careerpilot\ml


In [7]:
df = pd.read_csv(CSV_PATH)

df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [9]:
df.shape

(2484, 4)

In [10]:
df.columns.tolist()

['ID', 'Resume_str', 'Resume_html', 'Category']

In [11]:
df.isna().sum()

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

In [12]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [13]:
df["ID"].nunique(), len(df)

(2484, 2484)

In [14]:
df[df["ID"].duplicated(keep=False)].sort_values("ID")

,ID,Resume_str,Resume_html,Category


In [15]:
sample = df.sample(1, random_state=42).iloc[0]

print("ID:", sample["ID"])
print("Category:", sample["Category"])
print()
print(str(sample["Resume_str"])[:2000])

ID: 99244405
Category: TEACHER

           Kpandipou    Koffi         Summary      Compassionate teaching professional delivering exemplary support and assistance to teachers and students. Display exceptional Communication and problem solving skills.  Experience in office administration and public speaking. Attentive and adaptable, skilled in management of classroom operations. Effective in leveraging student feedback to create dynamic lesson plans that address individual strengths and weaknesses.  Dedicated and responsive team leader with proven skills in classroom management, behavior modification and individualized support.  Personable with experience using relationship-building to cultivate positive client, staff and management connections. Highly-developed communicator with outstanding skills in complex problem-solving and conflict resolution.  High-performing Administrative Assistant offering experience working with diverse client base and delivering exceptional results. Polished

In [16]:
print(str(sample["Resume_html"])[:2000])

<div class="RNA skn-rbn1 fontsize fontface vmargins hmargins pagesize dynamicbg" id="document"> <div class="parent-container" id="CONTAINER_PARENT_0"> <div class="separator-left" id="CONTAINER_0"> <div class="section SECTION_NAME firstsection" id="SECTION_NAME3628f436-18d5-4254-a1bc-e4e3502a9852"> <div class="paragraph PARAGRAPH_NAME firstparagraph" id="PARAGRAPH_3628f436-18d5-4254-a1bc-e4e3502a9852_1"> <div class="namebox" itemprop="name"> <span class="field" id="3628f436-18d5-4254-a1bc-e4e3502a9852SUBSTR_LNAM1"> </span> </div> <div class="name"> <span class="field" id="3628f436-18d5-4254-a1bc-e4e3502a9852FNAM1"> Kpandipou</span> <span> </span> <span class="field" id="3628f436-18d5-4254-a1bc-e4e3502a9852LNAM1"> Koffi</span> </div> </div> </div> </div> <div class="separator-main" id="CONTAINER_1"> <div class="section" id="SECTION_SUMM9667199b-26b5-44b9-ad7c-fb6672f98854"> <div class="heading"> <div class="sectiontitle" id="SECTNAME_SUMM9667199b-26b5-44b9-ad7c-fb6672f98854"> Summary</di

In [17]:
def extract_html_blocks(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    for element in soup(["script", "style"]):
        element.decompose()

    blocks = []

    for element in soup.find_all(
        ["h1", "h2", "h3", "h4", "h5", "h6", "p", "li", "td"]
    ):
        text = " ".join(element.get_text(" ", strip=True).split())

        if text:
            blocks.append(text)

    return blocks

In [20]:
blocks = extract_html_blocks(sample["Resume_html"])

for index, block in enumerate(blocks[:30]):
    print(index, repr(block))

0 'Compassionate teaching professional delivering exemplary support and assistance to teachers and students. Display exceptional Communication and problem solving skills.'
1 'Experience in office administration and public speaking. Attentive and adaptable, skilled in management of classroom operations. Effective in leveraging student feedback to create dynamic lesson plans that address individual strengths and weaknesses.'
2 'Dedicated and responsive team leader with proven skills in classroom management, behavior modification and individualized support.'
3 'Personable with experience using relationship-building to cultivate positive client, staff and management connections. Highly-developed communicator with outstanding skills in complex problem-solving and conflict resolution.'
4 'High-performing Administrative Assistant offering experience working with diverse client base and delivering exceptional results. Polished in managing client relations, and managing vendor relationships.'
5

In [21]:
def normalize_id(value: object) -> str:
    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    return text

In [25]:
csv_ids = {
    normalize_id(value)
    for value in df["ID"].dropna()
}

pdf_ids = {
    normalize_id(path.stem)
    for path in PDF_DIR.glob("*.pdf")
}

len(csv_ids), len(pdf_ids)

(2484, 2484)

In [26]:
missing_pdfs = sorted(csv_ids - pdf_ids)
orphan_pdfs = sorted(pdf_ids - csv_ids)

print("Missing PDFs:", len(missing_pdfs))
print("Orphan PDFs:", len(orphan_pdfs))

Missing PDFs: 0
Orphan PDFs: 0


In [27]:
samples = df.sample(10, random_state=42)

for _, row in samples.iterrows():
    resume_id = normalize_id(row["ID"])
    blocks = extract_html_blocks(str(row["Resume_html"]))

    print("=" * 80)
    print("ID:", resume_id)
    print("Category:", row["Category"])
    print("PDF exists:", (PDF_DIR / f"{resume_id}.pdf").exists())
    print("Resume_str length:", len(str(row["Resume_str"])))
    print("HTML block count:", len(blocks))
    print("First blocks:")

    for block in blocks[:10]:
        print("-", block)

ID: 99244405
Category: TEACHER
PDF exists: True
Resume_str length: 5576
HTML block count: 49
First blocks:
- Compassionate teaching professional delivering exemplary support and assistance to teachers and students. Display exceptional Communication and problem solving skills.
- Experience in office administration and public speaking. Attentive and adaptable, skilled in management of classroom operations. Effective in leveraging student feedback to create dynamic lesson plans that address individual strengths and weaknesses.
- Dedicated and responsive team leader with proven skills in classroom management, behavior modification and individualized support.
- Personable with experience using relationship-building to cultivate positive client, staff and management connections. Highly-developed communicator with outstanding skills in complex problem-solving and conflict resolution.
- High-performing Administrative Assistant offering experience working with diverse client base and delivering